# 02 — KG instantiation (validation & starter)

Validates that `kg/schema/ontology.ttl` actually works by using it: loads the
TBox, instantiates real POI data against it for a few sources (one per distinct
modelling pattern), runs SPARQL queries against the result, and leaves a
reusable helper + a fill-in-the-blanks section so the remaining sources can be
added the same way.

Three worked examples, chosen to cover the three distinct patterns from
`docs/kg_schema_design.md`:
1. **Museen** — the plain case: name, district, coordinates, address
2. **Parkanlagen** — adds `schema:amenityFeature` for boolean facts (dog-friendly,
   playground, water) plus a numeric `viennakg:areaSqm`
3. **Spielplätze** — the "separate nodes per feature" + equipment-list pattern

Everything else (Büchereien, Badestellen, Schwimmbäder, Sights, Wiener Linien
transport data) follows the same recipe — see the last section.

## Setup

In [11]:
import pandas as pd
import re
from rdflib import Graph, Namespace, URIRef, Literal, RDF
from rdflib.namespace import RDFS, XSD

In [12]:
VIENNAKG = Namespace("http://example.org/viennakg#")
SCHEMA = Namespace("https://schema.org/")
GEO = Namespace("http://www.w3.org/2003/01/geo/wgs84_pos#")

g = Graph()
g.parse("../kg/schema/ontology.ttl", format="turtle")
g.bind("viennakg", VIENNAKG)
g.bind("schema", SCHEMA)
g.bind("geo", GEO)

print("Loaded TBox:", len(g), "triples")

Loaded TBox: 161 triples


## Shared helpers

`safe_id` turns any messy source ID (e.g. `"MUSEUMOGD.138686"`) into a URI-safe
local name. `parse_point` extracts the first `lon lat` pair out of a WKT `SHAPE`
string — same logic used throughout the EDA notebooks.

In [13]:
def safe_id(raw) -> str:
    return re.sub(r"[^A-Za-z0-9_]", "_", str(raw))

def parse_point(shape):
    m = re.search(r"(-?\d+\.\d+)\s+(-?\d+\.\d+)", str(shape))
    return (float(m.group(1)), float(m.group(2))) if m else (None, None)

def bezirk_uri(bezirk):
    """BEZIRK int -> the matching viennakg:BezirkN individual, or None if missing/unmappable."""
    if pd.isnull(bezirk):
        return None
    return VIENNAKG[f"Bezirk{int(bezirk)}"]

## 1. Museen → `schema:Museum`

The plain case: one triple block per row for type, name, district, coordinates,
address, and an optional website.

In [14]:
museum = pd.read_csv("../data/processed/MUSEUMOGD_clean.csv")

for i, row in museum.iterrows():
    poi = VIENNAKG[f"museum_{safe_id(row['FID'])}"]
    lon, lat = parse_point(row["SHAPE"])

    g.add((poi, RDF.type, SCHEMA.Museum))
    g.add((poi, SCHEMA.name, Literal(row["NAME"])))
    g.add((poi, SCHEMA.address, Literal(row["ADRESSE"])))
    if lon is not None:
        g.add((poi, GEO.long, Literal(lon, datatype=XSD.decimal)))
        g.add((poi, GEO.lat, Literal(lat, datatype=XSD.decimal)))
    bez = bezirk_uri(row["BEZIRK"])
    if bez is not None:
        g.add((poi, SCHEMA.containedInPlace, bez))
    if pd.notnull(row.get("WEITERE_INF")):
        g.add((poi, SCHEMA.url, Literal(row["WEITERE_INF"])))

print(f"Instantiated {len(museum)} Museum POIs. Graph size now: {len(g)} triples")

Instantiated 136 Museum POIs. Graph size now: 1111 triples


## 2. Parkanlagen → `schema:Park`

Adds the `schema:amenityFeature` pattern: each Ja/Nein column becomes its own
`schema:LocationFeatureSpecification` blank node with a `schema:name` and a
boolean `schema:value` — rather than three separate ad-hoc boolean properties.

In [15]:
from rdflib import BNode

parks = pd.read_csv("../data/processed/PARKINFOOGD_clean.csv")

AMENITY_MAP = {
    "SPIELEN_IM_PARK": "Playground",
    "WASSER_IM_PARK": "Water feature",
    "HUNDE_IM_PARK": "Dogs allowed",
}

for i, row in parks.iterrows():
    poi = VIENNAKG[f"park_{safe_id(row['FID'])}"]
    lon, lat = parse_point(row["SHAPE"])

    g.add((poi, RDF.type, SCHEMA.Park))
    g.add((poi, SCHEMA.name, Literal(row["ANL_NAME"])))
    if lon is not None:
        g.add((poi, GEO.long, Literal(lon, datatype=XSD.decimal)))
        g.add((poi, GEO.lat, Literal(lat, datatype=XSD.decimal)))
    bez = bezirk_uri(row["BEZIRK"])
    if bez is not None:
        g.add((poi, SCHEMA.containedInPlace, bez))
    if pd.notnull(row.get("FLAECHE_M2")):
        g.add((poi, VIENNAKG.areaSqm, Literal(row["FLAECHE_M2"], datatype=XSD.decimal)))

    for col, label in AMENITY_MAP.items():
        val = str(row.get(col, "")).strip().lower() == "ja"
        feature = BNode()
        g.add((poi, SCHEMA.amenityFeature, feature))
        g.add((feature, RDF.type, SCHEMA.LocationFeatureSpecification))
        g.add((feature, SCHEMA.name, Literal(label)))
        g.add((feature, SCHEMA.value, Literal(val, datatype=XSD.boolean)))

print(f"Instantiated {len(parks)} Park POIs. Graph size now: {len(g)} triples")

Instantiated 1051 Park POIs. Graph size now: 20029 triples


## 3. Spielplätze → `viennakg:PlaygroundArea`

The "separate nodes per feature" pattern: every row is its own
`PlaygroundArea` instance (matches the source data — no aggregation by name).
`SPIELPLATZ_DETAIL`'s comma-separated equipment list becomes one
`LocationFeatureSpecification` per item via `schema:amenityFeature`, reusing the
exact same pattern as the Park amenities above.

In [16]:
playgrounds = pd.read_csv("../data/raw/SPIELPLATZPUNKTOGD.csv")

for i, row in playgrounds.iterrows():
    poi = VIENNAKG[f"playground_{safe_id(row['FID'])}"]
    lon, lat = parse_point(row["SHAPE"])

    g.add((poi, RDF.type, VIENNAKG.PlaygroundArea))
    g.add((poi, SCHEMA.name, Literal(row["ANL_NAME"])))
    if lon is not None:
        g.add((poi, GEO.long, Literal(lon, datatype=XSD.decimal)))
        g.add((poi, GEO.lat, Literal(lat, datatype=XSD.decimal)))
    bez = bezirk_uri(row["BEZIRK"])
    if bez is not None:
        g.add((poi, SCHEMA.containedInPlace, bez))

    detail = row.get("SPIELPLATZ_DETAIL")
    if pd.notnull(detail):
        for item in [x.strip() for x in str(detail).split(",") if x.strip()]:
            feature = BNode()
            g.add((poi, SCHEMA.amenityFeature, feature))
            g.add((feature, RDF.type, SCHEMA.LocationFeatureSpecification))
            g.add((feature, SCHEMA.name, Literal(item)))
            g.add((feature, SCHEMA.value, Literal(True, datatype=XSD.boolean)))

print(f"Instantiated {len(playgrounds)} PlaygroundArea POIs. Graph size now: {len(g)} triples")

Instantiated 771 PlaygroundArea POIs. Graph size now: 40623 triples


## Validate with SPARQL

If the modelling actually works, these should return sensible results without
any special-casing per source — everything's queryable the same way regardless
of which of the three patterns produced it.

In [17]:
q = """
PREFIX schema: <https://schema.org/>
SELECT ?class (COUNT(?poi) AS ?n) WHERE {
    ?poi a ?class .
    FILTER(?class IN (schema:Museum, schema:Park, <http://example.org/viennakg#PlaygroundArea>))
}
GROUP BY ?class
"""
for row in g.query(q):
    print(row.n, g.qname(row["class"]))

136 schema:Museum
1051 schema:Park
771 viennakg:PlaygroundArea


In [18]:
q = """
PREFIX schema: <https://schema.org/>
PREFIX viennakg: <http://example.org/viennakg#>
SELECT ?name WHERE {
    ?poi a schema:Museum ;
         schema:containedInPlace viennakg:Bezirk1 ;
         schema:name ?name .
}
ORDER BY ?name
LIMIT 10
"""
print("Museums in Bezirk 1 (Innere Stadt), first 10:")
for row in g.query(q):
    print(" -", row.name)

Museums in Bezirk 1 (Innere Stadt), first 10:
 - 3D PicArt Museum
 - Albertina
 - Albertina modern
 - Beethoven Pasqualatihaus
 - Bezirksmuseum Innere Stadt
 - Dom Museum Wien
 - Ephesos-Museum
 - Esperantomuseum der Österreichischen Nationalbibliothek
 - Feuerwehrmuseum
 - Gemäldegalerie der Akademie der bildenden Künste


In [19]:
q = """
PREFIX schema: <https://schema.org/>
SELECT ?name WHERE {
    ?poi a schema:Park ;
         schema:name ?name ;
         schema:amenityFeature ?f .
    ?f schema:name "Dogs allowed" ;
       schema:value true .
}
LIMIT 10
"""
print("Dog-friendly parks, first 10:")
for row in g.query(q):
    print(" -", row.name)

Dog-friendly parks, first 10:
 - Erika-Morini-Park
 - PA Blériotgasse
 - Fridtjof-Nansen-Park
 - Ferdinand-Kaufmann-Platz
 - Andreas-Rett-Park
 - Prater - Rustenschacher
 - PA Donaustadtstraße
 - Vilma-Webenau-Park
 - GA Aspernstraße
 - PA Gaulgasse


## Save the demo graph

Writes the TBox + these three sources' instances out as one Turtle file — useful
to open in a Turtle-aware editor or load elsewhere to sanity-check by eye.

In [20]:
g.serialize(destination="../kg/instances_demo.ttl", format="turtle")
print("saved kg/instances_demo.ttl —", len(g), "triples total (TBox + 3 sources' ABox)")

saved kg/instances_demo.ttl — 40623 triples total (TBox + 3 sources' ABox)


## 4. Büchereien → `schema:Library`

Same shape as Museum, plus two extras: the six `OEFFNUNGSZEITEN1..6` columns get
concatenated into one `schema:openingHours` string (kept as free text, per the
decision in `docs/kg_schema_design.md` — structured hours parsing is a stretch
goal), and `TELEFON`/`EMAIL`/`WEBLINK1` map to their obvious schema.org
equivalents.

In [21]:
buechereien = pd.read_csv("../data/raw/BUECHEREIOGD.csv")
oeff_cols = [c for c in buechereien.columns if c.startswith("OEFFNUNGSZEITEN")]

for i, row in buechereien.iterrows():
    poi = VIENNAKG[f"buecherei_{safe_id(row['FID'])}"]
    lon, lat = parse_point(row["SHAPE"])

    g.add((poi, RDF.type, SCHEMA.Library))
    g.add((poi, SCHEMA.name, Literal(row["NAME"])))
    g.add((poi, SCHEMA.address, Literal(row["ADRESSE"])))
    if lon is not None:
        g.add((poi, GEO.long, Literal(lon, datatype=XSD.decimal)))
        g.add((poi, GEO.lat, Literal(lat, datatype=XSD.decimal)))
    bez = bezirk_uri(row["BEZIRK"])
    if bez is not None:
        g.add((poi, SCHEMA.containedInPlace, bez))

    hours = "; ".join(str(row[c]) for c in oeff_cols if pd.notnull(row[c]))
    if hours:
        g.add((poi, SCHEMA.openingHours, Literal(hours)))
    if pd.notnull(row.get("TELEFON")):
        g.add((poi, SCHEMA.telephone, Literal(row["TELEFON"])))
    if pd.notnull(row.get("EMAIL")):
        g.add((poi, SCHEMA.email, Literal(row["EMAIL"])))
    if pd.notnull(row.get("WEBLINK1")):
        g.add((poi, SCHEMA.url, Literal(row["WEBLINK1"])))

print(f"Instantiated {len(buechereien)} Library POIs. Graph size now: {len(g)} triples")

Instantiated 37 Library POIs. Graph size now: 40992 triples


## 5. Badestellen → `viennakg:BathingSite`

Simplest of the seven: no address in the source, and `BADEQUALITAET`/`TYP` are
deliberately skipped (per your call — water quality is consistently good enough
across Vienna that it's not a useful discriminating feature).

In [22]:
badestellen = pd.read_csv("../data/raw/BADESTELLENOGD.csv")

for i, row in badestellen.iterrows():
    poi = VIENNAKG[f"badestelle_{safe_id(row['FID'])}"]
    lon, lat = parse_point(row["SHAPE"])

    g.add((poi, RDF.type, VIENNAKG.BathingSite))
    g.add((poi, SCHEMA.name, Literal(row["BEZEICHNUNG"])))
    if lon is not None:
        g.add((poi, GEO.long, Literal(lon, datatype=XSD.decimal)))
        g.add((poi, GEO.lat, Literal(lat, datatype=XSD.decimal)))
    bez = bezirk_uri(row["BEZIRK"])
    if bez is not None:
        g.add((poi, SCHEMA.containedInPlace, bez))

print(f"Instantiated {len(badestellen)} BathingSite POIs. Graph size now: {len(g)} triples")

Instantiated 32 BathingSite POIs. Graph size now: 41150 triples


## 6. Schwimmbäder → `viennakg:SwimmingPool`

Static facility info only — the `AUSLASTUNG_*` live occupancy columns stay out
of scope here too, same reasoning as Badestellen's water quality: the ontology
already has `viennakg:OccupancyStatus`/`hasOccupancyStatus` ready for it, but
wiring up live data is a Reasoning/Service Layer task, not KG Modelling.

In [23]:
schwimmbaeder = pd.read_csv("../data/raw/SCHWIMMBADOGD.csv")

for i, row in schwimmbaeder.iterrows():
    poi = VIENNAKG[f"schwimmbad_{safe_id(row['FID'])}"]
    lon, lat = parse_point(row["SHAPE"])

    g.add((poi, RDF.type, VIENNAKG.SwimmingPool))
    g.add((poi, SCHEMA.name, Literal(row["NAME"])))
    g.add((poi, SCHEMA.address, Literal(row["ADRESSE"])))
    if lon is not None:
        g.add((poi, GEO.long, Literal(lon, datatype=XSD.decimal)))
        g.add((poi, GEO.lat, Literal(lat, datatype=XSD.decimal)))
    bez = bezirk_uri(row["BEZIRK"])
    if bez is not None:
        g.add((poi, SCHEMA.containedInPlace, bez))
    if pd.notnull(row.get("WEBLINK1")):
        g.add((poi, SCHEMA.url, Literal(row["WEBLINK1"])))

print(f"Instantiated {len(schwimmbaeder)} SwimmingPool POIs. Graph size now: {len(g)} triples")

Instantiated 46 SwimmingPool POIs. Graph size now: 41471 triples


## 7. Sights (Wien Tourismus subset) → `schema:TouristAttraction`

The one source that uses `hasCategory`: each row's `SUBCATEGORY_NAME` picks
which of the two SKOS concepts defined in the ontology to attach
(`subcat_Sehenswuerdigkeit` or `subcat_SchlossPalais`).

In [24]:
sights = pd.read_csv("../data/processed/WIENTOURISMUS_sights_clean.csv")

SUBCAT_TO_CONCEPT = {
    "Sehenswürdigkeit": VIENNAKG.subcat_Sehenswuerdigkeit,
    "Schloss & Palais": VIENNAKG.subcat_SchlossPalais,
}

for i, row in sights.iterrows():
    poi = VIENNAKG[f"sight_{safe_id(row['FID'])}"]
    lon, lat = parse_point(row["SHAPE"])

    g.add((poi, RDF.type, SCHEMA.TouristAttraction))
    g.add((poi, SCHEMA.name, Literal(row["NAME"])))
    if pd.notnull(row.get("STREET")):
        g.add((poi, SCHEMA.address, Literal(row["STREET"])))
    if lon is not None:
        g.add((poi, GEO.long, Literal(lon, datatype=XSD.decimal)))
        g.add((poi, GEO.lat, Literal(lat, datatype=XSD.decimal)))
    bez = bezirk_uri(row["BEZIRK"])
    if bez is not None:
        g.add((poi, SCHEMA.containedInPlace, bez))

    concept = SUBCAT_TO_CONCEPT.get(row["SUBCATEGORY_NAME"])
    if concept is not None:
        g.add((poi, VIENNAKG.hasCategory, concept))

print(f"Instantiated {len(sights)} TouristAttraction POIs. Graph size now: {len(g)} triples")

Instantiated 247 TouristAttraction POIs. Graph size now: 43196 triples


## 8. Wiener Linien transport → `Stop` / `Platform` / `Line`

Different shape from the POI sources: three files that join together instead of
one file mapping to one class. Semicolon-delimited (not comma), and
`haltestellen.csv` already has separate `WGS84_LAT`/`WGS84_LON` columns instead
of a WKT `SHAPE` string, so no `parse_point` needed here.

Order matters: Lines and Stops first (so their URIs exist), then Platforms,
which link to both via `hasPlatform` (inverse, added from the Stop side) and
`servedByLine`.

In [25]:
linien = pd.read_csv("../data/raw/wienerlinien-ogd-linien.csv", sep=";")
haltestellen = pd.read_csv("../data/raw/wienerlinien-ogd-haltestellen.csv", sep=";")
steige = pd.read_csv("../data/raw/wienerlinien-ogd-steige.csv", sep=";")

for _, row in linien.iterrows():
    line = VIENNAKG[f"line_{safe_id(row['LINIEN_ID'])}"]
    g.add((line, RDF.type, VIENNAKG.Line))
    g.add((line, SCHEMA.name, Literal(row["BEZEICHNUNG"])))
    if pd.notnull(row.get("VERKEHRSMITTEL")):
        g.add((line, VIENNAKG.mode, Literal(row["VERKEHRSMITTEL"])))

for _, row in haltestellen.iterrows():
    stop = VIENNAKG[f"stop_{safe_id(row['HALTESTELLEN_ID'])}"]
    g.add((stop, RDF.type, VIENNAKG.Stop))
    g.add((stop, SCHEMA.name, Literal(row["NAME"])))
    g.add((stop, GEO.long, Literal(row["WGS84_LON"], datatype=XSD.decimal)))
    g.add((stop, GEO.lat, Literal(row["WGS84_LAT"], datatype=XSD.decimal)))

print(f"Lines + Stops instantiated. Graph size now: {len(g)} triples")

Lines + Stops instantiated. Graph size now: 51623 triples


In [26]:
n_platforms = 0
n_no_rbl = 0

for _, row in steige.iterrows():
    platform = VIENNAKG[f"platform_{safe_id(row['STEIG_ID'])}"]
    stop = VIENNAKG[f"stop_{safe_id(row['FK_HALTESTELLEN_ID'])}"]
    line = VIENNAKG[f"line_{safe_id(row['FK_LINIEN_ID'])}"]

    g.add((platform, RDF.type, VIENNAKG.Platform))
    g.add((stop, VIENNAKG.hasPlatform, platform))
    g.add((platform, VIENNAKG.servedByLine, line))

    if pd.notnull(row.get("RBL_NUMMER")):
        g.add((platform, VIENNAKG.rbl, Literal(int(row["RBL_NUMMER"]), datatype=XSD.integer)))
        n_platforms += 1
    else:
        n_no_rbl += 1

print(f"Instantiated {len(steige)} Platforms ({n_platforms} with an RBL, {n_no_rbl} without).")
print(f"Graph size now: {len(g)} triples")

Instantiated 7362 Platforms (6776 with an RBL, 586 without).
Graph size now: 80485 triples


## Validate the full graph

Same style of check as before, now across all seven POI classes plus the
transport side.

In [27]:
q = """
PREFIX schema: <https://schema.org/>
PREFIX viennakg: <http://example.org/viennakg#>
SELECT ?class (COUNT(?x) AS ?n) WHERE {
    ?x a ?class .
    FILTER(?class IN (
        schema:Museum, schema:Library, viennakg:BathingSite, schema:Park,
        viennakg:SwimmingPool, viennakg:PlaygroundArea, schema:TouristAttraction,
        viennakg:Stop, viennakg:Platform, viennakg:Line
    ))
}
GROUP BY ?class
ORDER BY DESC(?n)
"""
for row in g.query(q):
    print(f"{int(row.n):>6}  {g.qname(row['class'])}")

  7362  viennakg:Platform
  1959  viennakg:Stop
  1051  schema:Park
   771  viennakg:PlaygroundArea
   247  schema:TouristAttraction
   197  viennakg:Line
   136  schema:Museum
    46  viennakg:SwimmingPool
    37  schema:Library
    32  viennakg:BathingSite


In [28]:
# Sanity check the transport join: pick one stop, list its platforms and lines
q = """
PREFIX schema: <https://schema.org/>
PREFIX viennakg: <http://example.org/viennakg#>
SELECT ?stopName ?rbl ?lineName WHERE {
    ?stop a viennakg:Stop ; schema:name ?stopName ; viennakg:hasPlatform ?platform .
    ?platform viennakg:servedByLine ?line ; viennakg:rbl ?rbl .
    ?line schema:name ?lineName .
}
ORDER BY ?stopName
LIMIT 10
"""
for row in g.query(q):
    print(f"{row.stopName!s:35} RBL {row.rbl!s:>6}  line {row.lineName}")

1. Haidequerstraße Mitte            RBL   5028  line 72A
1. Haidequerstraße Mitte            RBL   5028  line 72A
1. Haidequerstraße Mitte            RBL   5028  line 76B
1. Haidequerstraße Süd              RBL    449  line 72A
1. Haidequerstraße Süd              RBL    439  line 72A
1. Haidequerstraße Süd              RBL    449  line 76B
11. Haidequerstraße                 RBL   5003  line 76A
11. Haidequerstraße                 RBL   5024  line 76A
11. Haidequerstraße                 RBL   5003  line 76B
11. Haidequerstraße                 RBL   5024  line 76B


In [29]:
# Cross-source query: sights within Bezirk 22, tagged by subcategory
q = """
PREFIX schema: <https://schema.org/>
PREFIX viennakg: <http://example.org/viennakg#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT ?name ?subcat WHERE {
    ?poi a schema:TouristAttraction ;
         schema:name ?name ;
         schema:containedInPlace viennakg:Bezirk1 ;
         viennakg:hasCategory ?concept .
    ?concept skos:prefLabel ?subcat .
}
ORDER BY ?name
"""
print("Sights in Bezirk 1, with subcategory:")
for row in g.query(q):
    print(f" - {row.name} ({row.subcat})")

Sights in Bezirk 1, with subcategory:
 - 3. Mann Tour (Sehenswürdigkeit)
 - Adels Casino (Schloss & Palais)
 - Alte Universität (Sehenswürdigkeit)
 - Altes Rathaus (Sehenswürdigkeit)
 - Am Hof (Sehenswürdigkeit)
 - Ampelpärchen (Sehenswürdigkeit)
 - Ankerhaus (Sehenswürdigkeit)
 - Ankeruhr (Sehenswürdigkeit)
 - Anna Sacher (Sehenswürdigkeit)
 - Artaria-Haus (Sehenswürdigkeit)
 - Beethoven-Blick (Sehenswürdigkeit)
 - Constanze Geiger (Sehenswürdigkeit)
 - Deutschordenshaus (Sehenswürdigkeit)
 - Erlebnis Europa (Sehenswürdigkeit)
 - Freyung (Sehenswürdigkeit)
 - Haas-Haus (Sehenswürdigkeit)
 - Heiligenkreuzerhof (Sehenswürdigkeit)
 - Heldenplatz (Sehenswürdigkeit)
 - Hochhaus Herrengasse (Sehenswürdigkeit)
 - Hofburg (Sehenswürdigkeit)
 - Hohe Brücke (Sehenswürdigkeit)
 - Hoher Markt (Sehenswürdigkeit)
 - Hotel Imperial (Sehenswürdigkeit)
 - Judenplatz (Sehenswürdigkeit)
 - Jugendstil-Toilette am Graben (Sehenswürdigkeit)
 - Justizpalast (Sehenswürdigkeit)
 - Landespolizeidirektion – Süh

## Save the full graph

All 7 POI sources plus the Wiener Linien transport data, in one Turtle file.

In [30]:
g.serialize(destination="../kg/instances_demo.ttl", format="turtle")
print("saved kg/instances_demo.ttl —", len(g), "triples total (TBox + full ABox)")

saved kg/instances_demo.ttl — 80485 triples total (TBox + full ABox)


## Summary

Every source from `docs/kg_schema_design.md`'s mapping table is now
instantiated and cross-queryable in one graph: 7 POI classes (with all three
modelling patterns — plain, amenity-featured, and separate-nodes-per-feature),
plus the Wiener Linien Stop/Platform/Line join. Live data (`Departure`,
`Disruption`, `OccupancyStatus`) stays unpopulated for now, this can be added later 
in a separate Reasoning/Service Layer step if time